SO previously i already attempted doing 

In [1]:
import os 
import torch 
import numpy as np
from PIL import Image
from torch.utils.data import Dataset,DataLoader,random_split
from torchvision import transforms,models

In [2]:
base_path = "bottle"
train_path = os.path.join(base_path,"train","good")
test_path = os.path.join(base_path,"test")
print("Train path",train_path)
print("Test path",test_path)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


Train path bottle\train\good
Test path bottle\test


In [3]:
class NormalBottleDataset(Dataset):

    def __init__(self, folder, transform=None):
        self.folder = folder
        self.transform = transform
        self.images = os.listdir(folder)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):

        image_name = self.images[index]

        image_path = os.path.join(
            self.folder,
            image_name
        )

        image = Image.open(
            image_path
        ).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image



In [4]:
full_dataset = NormalBottleDataset(train_path,transform=transform)

In [5]:
train_size = int(
    0.8 * len(full_dataset)
)

val_size = (
    len(full_dataset) - train_size
)

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [7]:
print("Total normal images:", len(full_dataset))
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))

Total normal images: 209
Training images: 167
Validation images: 42


In [8]:
from torchvision.models import resnet18, ResNet18_Weights
weights = ResNet18_Weights.DEFAULT
resnet = resnet18(weights=weights)

In [9]:
feature_extractor = torch.nn.Sequential(
    *list(resnet.children())[:-1]
)

In [10]:
feature_extractor.eval()
for param in feature_extractor.parameters():
    param.requires_grad = False
images = next(iter(train_loader))
with torch.no_grad():
    features = feature_extractor(images)
print("Images shape:", images.shape)
print("Features shape:", features.shape)

Images shape: torch.Size([32, 3, 224, 224])
Features shape: torch.Size([32, 512, 1, 1])


In [11]:
all_features = []
feature_extractor.eval()
with torch.no_grad():
    for image in train_loader:
        features = feature_extractor(image)
        features = features.flatten(1)
        all_features.append(features)

train_features = torch.cat(
    all_features,dim=0
)
print(train_features.shape)

torch.Size([167, 512])


In [ ]:
# ============================================
# VALIDATION: CALCULATE ANOMALY SCORES
# ============================================

all_val_scores = []

feature_extractor.eval()

with torch.no_grad():

    for val_images in val_loader:

        # 1. Pass validation images through ResNet18
        val_features = feature_extractor(val_images)

        val_features = val_features.flatten(1)

        # 3. Compare validation features
        #    with all normal training features
        distances = torch.cdist(
            val_features,
            train_features
        )

        # 4. For each validation image,
        #    take distance to nearest normal training image
        min_distances, _ = torch.min(
            distances,
            dim=1
        )

        # 5. Store anomaly scores
        all_val_scores.extend(
            min_distances.cpu().numpy()
        )

In [15]:
print("Number of validation scores:", len(all_val_scores))

print("Mean:", np.mean(all_val_scores))
print("Maximum:", np.max(all_val_scores))
print("95th percentile:", np.percentile(all_val_scores, 95))
print("99th percentile:", np.percentile(all_val_scores, 99))

Number of validation scores: 42
Mean: 3.377129
Maximum: 4.5456934
95th percentile: 4.2809978
99th percentile: 4.5297155


In [16]:
threshold = np.percentile(
    all_val_scores,
    99
)

print("Threshold:", threshold)

Threshold: 4.5297155


In [19]:
import os
import torch
from PIL import Image
y_true = []
y_pred = []
test_scores = []
feature_extractor.eval()
with torch.no_grad():
    for folder in os.listdir(test_path):
        folder_path = os.path.join(test_path, folder)
        actual_label = 0 if folder == "good" else 1
        for image_name in os.listdir(folder_path):
            image_path = os.path.join(
                folder_path,
                image_name
            )

            image = Image.open(
                image_path
            ).convert("RGB")

            image = transform(image)
            image = image.unsqueeze(0)
            features = feature_extractor(image)
            features = features.flatten(1)
            distances = torch.cdist(
                features,
                train_features
            )
            anomaly_score = torch.min(
                distances
            ).item()
            predicted_label = (
                1 if anomaly_score > threshold else 0
            )

            y_true.append(actual_label)
            y_pred.append(predicted_label)
            test_scores.append(anomaly_score)


In [20]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score
)

print("Confusion Matrix:")
print(
    confusion_matrix(
        y_true,
        y_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        target_names=[
            "Normal",
            "Defective"
        ]
    )
)

roc_auc = roc_auc_score(
    y_true,
    test_scores
)

print("ROC-AUC:", roc_auc)

Confusion Matrix:
[[19  1]
 [ 5 58]]

Classification Report:
              precision    recall  f1-score   support

      Normal       0.79      0.95      0.86        20
   Defective       0.98      0.92      0.95        63

    accuracy                           0.93        83
   macro avg       0.89      0.94      0.91        83
weighted avg       0.94      0.93      0.93        83

ROC-AUC: 0.9904761904761905


In [21]:
import torch

torch.save({
    "train_features": train_features,
    "threshold": threshold
}, "bottle_anomaly_data.pth")

print("Saved successfully")

Saved successfully


In [22]:
from PIL import Image
import torch

def predict_image(image_path):

    feature_extractor.eval()

    # Load image
    image = Image.open(image_path).convert("RGB")

    # Apply preprocessing
    image = transform(image)

    # Add batch dimension
    image = image.unsqueeze(0)

    with torch.no_grad():

        # Extract 512 features
        features = feature_extractor(image)

        features = features.flatten(1)

        # Compare with normal feature bank
        distances = torch.cdist(
            features,
            train_features
        )

        # Nearest normal image
        anomaly_score = torch.min(
            distances
        ).item()

    # Final decision
    if anomaly_score > threshold:
        prediction = "Defective"
    else:
        prediction = "Normal"

    return {
        "prediction": prediction,
        "anomaly_score": anomaly_score,
        "threshold": threshold
    }

In [24]:
result = predict_image(
    "bottle/test/broken_large/000.png"
)

print(result)

{'prediction': 'Defective', 'anomaly_score': 9.03490161895752, 'threshold': np.float32(4.5297155)}
